In [ ]:
import espressomd.electrostatics
import espressomd.electrostatic_extensions

pw_error = 1e-6
prefactor = 1.0

system = espressomd.System(box_l=[10, 10, 3])
system.time_step = 0.01

p3m = espressomd.electrostatics.P3M(
        prefactor=prefactor, accuracy=pw_error, check_neutrality=False, verbose=False
    )
icc = espressomd.electrostatic_extensions.ICC(n_icc=10)
system.electrostatics.solver = p3m
system.electrostatics.extension = icc

# Set the ICC line density and calculate the number of
# ICC particles according to the box size
box_l = 9.
system.box_l = [box_l, box_l, 12.]
nicc = 3  # linear density
nicc_per_electrode = nicc**2  # surface density
nicc_tot = 2 * nicc_per_electrode
iccArea = box_l**2 / nicc_per_electrode
l = box_l / nicc

# Lists to collect required parameters
iccNormals = []
iccAreas = []
iccSigmas = []
iccEpsilons = []

# Add the fixed ICC particles:

# Left electrode (normal [0, 0, 1])
for xi in range(nicc):
    for yi in range(nicc):
        system.part.add(pos=[l * xi, l * yi, 0.], q=-0.0001,
                        type=icc_type, fix=[True, True, True])
iccNormals.extend([0, 0, 1] * nicc_per_electrode)

# Right electrode (normal [0, 0, -1])
for xi in range(nicc):
    for yi in range(nicc):
        system.part.add(pos=[l * xi, l * yi, box_l], q=0.0001,
                        type=icc_type, fix=[True, True, True])
iccNormals.extend([0, 0, -1] * nicc_per_electrode)

# Common area, sigma and metallic epsilon
iccAreas.extend([iccArea] * nicc_tot)
iccSigmas.extend([0] * nicc_tot)
iccEpsilons.extend([100000] * nicc_tot)

icc = espressomd.electrostatic_extensions.ICC(
    first_id=0,
    n_icc=nicc_tot,
    convergence=1e-4,
    relaxation=0.75,
    ext_field=[0, 0, 0],
    max_iterations=100,
    eps_out=1.0,
    normals=iccNormals,
    areas=iccAreas,
    sigmas=iccSigmas,
    epsilons=iccEpsilons)

system.electrostatics.extension = icc